<a href="https://colab.research.google.com/github/obscure-n8/Wzmlanddzone/blob/main/zxzone_public_deploy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1>ZxZone Heroku Deploy</h1>

<center><img src='https://raw.githubusercontent.com/obscure-n8/Wzmlanddzone/main/zxzone.png'  height="200" width="400" alt="ZxZone"/></center>

---

### Deploy Details
- 🔗 **Upstream Repo :** https://github.com/obscure-n8/Wzmlanddzone
- 📦 **Deploy Repo :** https://github.com/DownloaderZone/WZ-Deploy
- 🏷 **Version :** _v1.2.0_
- 👤 **Code Owner :** https://github.com/obscure-n8

---
### Deploy ZxZone on Heroku using Google Colab

> **Note:** This notebook is public. Anyone can use it to deploy their own ZxZone bot on Heroku.

---

### Built-in Optimizations
- `pyrogram==2.0.106` forced (stable)
- `wzgram` removed (lighter)
- `MEM_BUDGET=400` (Heroku 512MB safe)
- `CPU_LIMIT=20`
- `THROTTLE_SERVICES=always`
- `BOT_MAX_TASKS=3`
- `STATUS_UPDATE_INTERVAL=10`
- `FFMPEG_CORES=0` (single core safe)
- Auto-restart protection
- Heroku anti-ban soft restart

In [ ]:
#@title <center><h3>***Heroku Login***</h3></center><br>

#@markdown ---

Heroku_Email = "" #@param {type:"string"}
Heroku_API = "" #@param {type:"string"}
#@markdown <h6>( <b>Note:</b> <i>Team App Deploy, Use your Personal API Token</i> )</h6>

#@markdown ---

!curl -s https://cli-assets.heroku.com/install.sh | sh

from IPython.display import HTML, clear_output, display
clear_output()
display(HTML("<marquee><b>Heroku CLI Installed !</b></marquee>"))

if not all([Heroku_Email, Heroku_API]):
    raise ValueError("Please fill in all Mandatory Variables.")

from os import path as ospath, chmod

netrc_path = ospath.expanduser("~/.netrc")

netrc_creds = f'''machine api.heroku.com
  login {Heroku_Email}
  password {Heroku_API}
machine git.heroku.com
  login {Heroku_Email}
  password {Heroku_API}'''
with open(netrc_path, "w") as netrc_file:
    netrc_file.write(netrc_creds)

chmod(netrc_path, 0o600)

!git config --global user.email {Heroku_Email}
!git config --global user.name "ZxZone"

display(HTML("<marquee><b>Heroku Email & Password Loaded!</b></marquee>"))

In [ ]:
#@title <center><h3>***Create Heroku App(s)***</h3></center><br>

#@markdown ---

App_Names = "" #@param {type:"string"}
#@markdown <h6>( <b>Syntax:</b> <i>bot_name1 bot_name2, separated by space !</i> )</h6>
#@markdown <h6>( <b>Note:</b> <i>App Name is Optional, skip for random name !</i> )</h6>

Server_Region = "eu" #@param ["eu", "us"] {allow-input: true}
HK_Team_Name = "" #@param {type:"string"}
#@markdown <h6>( <b>Note:</b> <i>Team App Deploy, Optional option only if you want to deploy to Teams !</i> )</h6>

#@markdown ---

HK_Team_Name = f"--team {HK_Team_Name}" if HK_Team_Name else ""
for App_Name in App_Names.split():
    !heroku create --region $Server_Region --stack container $HK_Team_Name $App_Name

In [ ]:
#@title <center><h3>***ZxZone Repo Config Setup***</h3></center><br>

#@markdown ---
App_Name = "" #@param {type:"string"}
#@markdown <h6>( <b>Note:</b> <i>Config Setup for this App Name, Change the App Name for every Config Save!</i> )</h6>

#@markdown ---

#@markdown #### ***Fill all these Variables for the `config.py`*** **(All are Mandatory)**

BOT_TOKEN = ""  # @param {type:"string"}
TELEGRAM_API = 0  # @param {type:"integer"}
TELEGRAM_HASH = ""  # @param {type:"string"}
OWNER_ID = 0  # @param {type:"integer"}
UPSTREAM_REPO = "https://github.com/obscure-n8/Wzmlanddzone"  # @param {type:"string"}
UPSTREAM_BRANCH = "main"  # @param {type:"string"}
DATABASE_URL = ""  # @param {type:"string"}
BASE_URL = ""  # @param {type:"string"}

#@markdown ---

#@markdown ### ***OR***
CONF_GIST_URL = "" # @param {type:"string"}
#@markdown <h6><i>(If you want to Upload `config.py` via `gist.github.com`, Provide the gist URL, Always make Private gist)</i></h6>

#@markdown ---

#@markdown ### ***OR***
Upload_Config = False # @param {type:"boolean"}
#@markdown <h6><i>(If you want to Upload `config.py` file Externally, Tick the Above CheckBox)</i></h6>

#@markdown ---

#@markdown ### ***Built-in Heroku Optimization (Pre-filled, do not change unless needed)***
Modify_Pkgs = "pyrogram==2.0.106" #@param {type:"string"}
Remove_Pkgs = "wzgram" #@param {type:"string"}
#@markdown <h6><i>(Modify / Remove packages before deploy — keeps the bot light on Heroku)</i></h6>

#@markdown ---

#@markdown ### ***Extra Heroku-Safe Env Vars (Auto-injected)***
Set_Extra_Env = True #@param {type:"boolean"}
#@markdown <h6><i>(Injects MEM_BUDGET, CPU_LIMIT, THROTTLE_SERVICES, BOT_MAX_TASKS, STATUS_UPDATE_INTERVAL, FFMPEG_CORES)</i></h6>

#@markdown ---

if not App_Name and not Upload_Config and not CONF_GIST_URL and not all([BOT_TOKEN, TELEGRAM_API, TELEGRAM_HASH, OWNER_ID, UPSTREAM_REPO, DATABASE_URL, BASE_URL]):
    raise ValueError("Please fill in all Mandatory Variables.")

from os import path, remove

if path.isdir(App_Name):
    !rm -rf $App_Name

!git clone https://github.com/DownloaderZone/WZ-Deploy $App_Name
%cd $App_Name

for file in ["README.md", "wzv3_hk_deploy.ipynb", "zxzone_public_deploy.ipynb"]:
    if path.isfile(file):
        remove(file)

# Pull bot source from upstream repo (main branch)
!git remote add upstream https://github.com/obscure-n8/Wzmlanddzone
!git fetch upstream main
!git checkout upstream/main -- . 2>/dev/null || git checkout upstream/main .

# Restore deploy-specific files from WZ-Deploy (Dockerfile, requirements, heroku.yml, Procfile, alive.py, app.json, runtime.txt)
!git checkout HEAD -- Dockerfile requirements.txt .gitignore .dockerignore heroku.yml Procfile alive.py app.json runtime.txt config_sample.env 2>/dev/null || true

if Upload_Config:
    from google.colab import files
    config_creds = list(files.upload().values())[0]
    with open("config.py", "wb") as config_file:
        config_file.write(config_creds)
    print("config.py File Uploaded and Saved Successfully")
elif CONF_GIST_URL:
    !curl -o "config.py" $CONF_GIST_URL
    print("config.py File Extracted and Saved Successfully")
elif all([BOT_TOKEN, TELEGRAM_API, TELEGRAM_HASH, OWNER_ID, UPSTREAM_REPO, DATABASE_URL, BASE_URL]):
    extra_env = ''
    if Set_Extra_Env:
        extra_env = (
            f'MEM_BUDGET = 400\n'
            f'CPU_LIMIT = 20\n'
            f'THROTTLE_SERVICES = "always"\n'
            f'BOT_MAX_TASKS = 3\n'
            f'STATUS_UPDATE_INTERVAL = 10\n'
            f'FFMPEG_CORES = 0\n'
            f'MEM_DEEP_STATS = False\n'
            f'ENABLE_TELEMETRY = False\n'
        )
    config_creds = f'BOT_TOKEN = "{BOT_TOKEN}"\n' \
                   f'TELEGRAM_API = {TELEGRAM_API}\n' \
                   f'TELEGRAM_HASH = "{TELEGRAM_HASH}"\n' \
                   f'OWNER_ID = {OWNER_ID}\n' \
                   f'UPSTREAM_REPO = "{UPSTREAM_REPO}"\n' \
                   f'UPSTREAM_BRANCH = "{UPSTREAM_BRANCH}"\n' \
                   f'DATABASE_URL = "{DATABASE_URL}"\n' \
                   f'BASE_URL = "{BASE_URL}"\n' \
                   f'{extra_env}'

    with open("config.py", "wb") as config_file:
        config_file.write(config_creds.encode())
    print("config.py File made and Saved Successfully")

if Modify_Pkgs or Remove_Pkgs:
    from re import split as rsplit, escape
    edit_pkgs = [pkg.strip() for pkg in Modify_Pkgs.split(',') if pkg.strip()]
    rm_pkgs = [pkg.strip() for pkg in Remove_Pkgs.split(',') if pkg.strip()]

    with open("requirements.txt", "r") as req_file:
        contents = req_file.readlines()

    new_contents = []
    for line in contents:
        skip = False
        for pkg in rm_pkgs:
            if line.lower().startswith(pkg.lower()):
                skip = True
                break
        if not skip:
            new_contents.append(line)
    contents = new_contents

    for i, line in enumerate(contents):
        for pkg in edit_pkgs[:]:
            base = rsplit('|'.join(map(escape, ['==', '>=', '<=', '~='])), pkg)[0]
            if line.lower().startswith(base.lower()):
                contents[i] = f"{pkg}\n"
                edit_pkgs.remove(pkg)

    for pkg in edit_pkgs:
        contents.append(f"{pkg}\n")

    with open("requirements.txt", "w") as req_file:
        req_file.writelines(contents)
    print(f"requirements.txt updated : {Modify_Pkgs} | removed : {Remove_Pkgs}")

%cd ..

print("All Available Config Bot Names Saved :")
!ls

In [ ]:
#@title <center><h3>***Deploy Heroku App(s)***</h3></center><br>

#@markdown ---
App_Names = "" #@param {type:"string"}
#@markdown <h6>( <b>Syntax:</b> <i>bot_name1 bot_name2, separated by space ! Config must be set for that bot to deploy</i> )</h6>

#@markdown ---

from os import path as ospath

for App_Name in App_Names.split():
    if ospath.isdir(App_Name):
        %cd $App_Name
        !git add . -f
        !git commit -m "ZxZone HK Setup"
        !heroku git:remote -a $App_Name
        !git push heroku main -f
        %cd ..
    else:
        print(f"Config not set for {App_Name}, Again run the previous cell with App Name")

In [ ]:
#@title <center><h3>***Set Extra Heroku Config Vars***</h3></center><br>

#@markdown ---
App_Name = "" #@param {type:"string"}
#@markdown <h6>( <b>Note:</b> <i>Push Heroku-safe env vars directly into the Heroku app config</i> )</h6>

#@markdown ---

if App_Name:
    !heroku config:set MEM_BUDGET=400 -a {App_Name}
    !heroku config:set CPU_LIMIT=20 -a {App_Name}
    !heroku config:set THROTTLE_SERVICES=always -a {App_Name}
    !heroku config:set BOT_MAX_TASKS=3 -a {App_Name}
    !heroku config:set STATUS_UPDATE_INTERVAL=10 -a {App_Name}
    !heroku config:set FFMPEG_CORES=0 -a {App_Name}
    !heroku config:set ENABLE_TELEMETRY=False -a {App_Name}
    print(f"Heroku-safe env vars set for {App_Name}")
else:
    print("App_Name is empty")

In [ ]:
#@title <center><h3>***Scale Heroku Dyno***</h3></center><br>

#@markdown ---
App_Name = "" #@param {type:"string"}
Dyno_Type = "worker" #@param ["worker", "web"]
Quantity = 1 #@param {type:"integer"}
Size = "eco" #@param ["eco", "basic", "standard-1x", "standard-2x"]
#@markdown ---

if App_Name:
    !heroku ps:scale {Dyno_Type}={Quantity}:{Size} -a {App_Name}
    print(f"Scaled {App_Name} — {Dyno_Type}={Quantity}:{Size}")
else:
    print("App_Name is empty")

In [ ]:
#@title <center><h3>***Show Heroku App Logs***</h3></center><br>

#@markdown ---
App_Name = "" #@param {type:"string"}
#@markdown ---

!heroku logs -t -a {App_Name}

In [ ]:
#@title <center><h3>***Restart Heroku App***</h3></center><br>

#@markdown ---
App_Name = "" #@param {type:"string"}
#@markdown ---

!heroku restart -a {App_Name}

In [ ]:
#@title <center><h3>***Heroku Logout***</h3></center><br>

!heroku logout